In [ ]:
import numpy as np
import gymnasium as gym
from stable_baselines3 import A2C
import os


In [34]:
models_dir = "./models"
os.makedirs(models_dir, exist_ok=True)

a2c_path = os.path.join(models_dir, "a2c")

data_dir = "./data"
a2c_data_dir = os.path.join(data_dir, "a2c")
os.makedirs(a2c_data_dir, exist_ok=True)

x_path = os.path.join(a2c_data_dir, "X_cartpole.npy")
y_path = os.path.join(a2c_data_dir, "y_cartpole.npy")


In [35]:
env = gym.make("CartPole-v1")


In [36]:
# Cargamos el mejor modelo guardado por EvalCallback

best_model = A2C.load(a2c_path)
print("Mejor modelo cargado.")


Mejor modelo cargado.


c:\Users\malos\Documents\GitHub\XRL\.venv\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run A2C on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [37]:
def evaluar_politica_modelo(env: gym.Env, model: A2C, n_episodes: int = 20, deterministic: bool = True):
    rewards = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done, total = False, 0.0
        while not done:
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, r, term, trunc, _ = env.step(int(action))
            total += r
            done = term or trunc
        rewards.append(total)
    return np.mean(rewards), np.std(rewards)

m, s = evaluar_politica_modelo(env, best_model)
print(f"Recompensa del A2C: {m:.1f} ± {s:.1f}")


Recompensa del A2C: 500.0 ± 0.0


In [38]:
def recolectar_dataset(env: gym.Env, model: A2C, n_episodes: int = 200, deterministic: bool = True):
    X, y = [], []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        while not done:
            action, _ = model.predict(obs, deterministic=deterministic)
            X.append(obs)
            y.append(int(action))
            obs, _, terminated, truncated, _ = env.step(int(action))
            done = terminated or truncated
    return np.array(X), np.array(y)

X, y = recolectar_dataset(env, best_model, n_episodes=200)
print("Dataset:", X.shape, y.shape)
print("Distribución de acciones:", np.bincount(y))


Dataset: (100000, 4) (100000,)
Distribución de acciones: [49990 50010]


In [39]:
np.save(x_path, X)
np.save(y_path, y)
